## 1. External Imports

In [ ]:
# Standard library
import sys
import os
import io
import math
import random
from fractions import Fraction

# Third-party
import numpy as np
import matplotlib.pyplot as plt

# Qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import UnitaryGate, MCPhaseGate
from qiskit.quantum_info import Operator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Aer (local simulator, used as a fallback / for Phase 2's noiseless demo)
from qiskit_aer import AerSimulator

# Qiskit IBM Runtime (real hardware)
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2


## 2. Customization

In [ ]:
IBM_API_KEY = "Insert thy API Key, King"
IBM_INSTANCE_CRN = ""  # leave blank if you don't have/need an instance CRN

# Valid tests, try to increase these [57, 95, 111, 123]
def GiveN():
    N_list = [95]
    return random.choice(N_list)


## 3. Internal Utilities

In [ ]:
def GiveHammingWeight(x):
    """Number of 1-bits in the binary representation of x."""
    BitX = bin(x)[2:]
    return BitX.count("1")


In [ ]:
def CreateDimension(N):
    """Return a random non-trivial [factor, cofactor] pair of N (or None if N is prime)."""
    DimList = []
    for i in range(2, int(N**0.5) + 1):
        if N % i == 0:
            DimList.append([i, N // i])
    if not DimList:
        return None
    return random.choice(DimList)


In [ ]:
def GenerateKey(Dimension):
    """Pick a random (x, y) pair, x even, whose Hamming-weight XOR matches
    HW(Width) + HW(Height). Used by Phase 2 as the Grover search target."""
    x, y = Dimension
    KeyList = []

    TargetWeightSum = GiveHammingWeight(x) + GiveHammingWeight(y)

    for i in range(0, x, 2):  # even x values only
        weight_i = GiveHammingWeight(i)
        for j in range(y):
            if (weight_i ^ GiveHammingWeight(j)) == TargetWeightSum:
                KeyList.append([i, j])

    if KeyList:
        return random.choice(KeyList)

    return None


In [ ]:
def CreateSpace(N):
    w, h = CreateDimension(N)
    QRWidth = QuantumRegister(w, name="Width")
    QRHeight = QuantumRegister(h, name="Height")
    QC = QuantumCircuit(QRWidth, QRHeight)
    return QC, [w, h]


def CreateSpaceDimensioned(Width, Height):
    QRWidth = QuantumRegister(Width, name="Width")
    QRHeight = QuantumRegister(Height, name="Height")
    QC = QuantumCircuit(QRWidth, QRHeight)
    return QC, QRWidth, QRHeight


## 4. Classical Reference Check

For verification/debugging only — **do not use this on the hardware run**,
it defeats the point of using Shor's algorithm

In [ ]:
def find_order_classically(a, N):
    """Classical reference implementation. Finds the smallest positive r
    such that a^r = 1 mod N. ONLY for verification."""
    value = 1
    for r in range(1, N + 1):
        value = (value * a) % N
        if value == 1:
            return r
    return None


## 5. Phase 1 Building Blocks

In [ ]:
def ChooseGamma(N):
    """Pick a random base 'a' (here called Gamma) coprime to N."""
    GammaList = [i for i in range(N) if i > 1 and math.gcd(i, N) == 1]
    return random.choice(GammaList)


In [ ]:
def BuildUnitary(Gamma, N):
    """Build the permutation unitary U|x> = |Gamma*x mod N> (for x coprime to N),
    padded out to a full 2^n x 2^n permutation matrix."""
    if math.gcd(Gamma, N) != 1:
        raise ValueError(f"Gamma ({Gamma}) and N ({N}) are not coprime! Matrix cannot be unitary.")

    num_qubits = int(np.ceil(np.log2(N)))
    dim = 2 ** num_qubits

    permutation = list(range(dim))
    mapped_destinations = set()

    # Map coprime elements via modular multiplication
    for x in range(N):
        if math.gcd(x, N) == 1:
            next_state = (x * Gamma) % N
            permutation[x] = next_state
            mapped_destinations.add(next_state)

    # Bridge unmapped indices to keep the permutation 1-to-1
    available_destinations = [d for d in range(dim) if d not in mapped_destinations]
    avail_idx = 0
    for x in range(dim):
        if x >= N or math.gcd(x, N) != 1:
            permutation[x] = available_destinations[avail_idx]
            avail_idx += 1

    matrix = np.zeros((dim, dim), dtype=np.float64)
    for src, dest in enumerate(permutation):
        matrix[dest, src] = 1.0

    return Operator(matrix)


def power_of_unitary(U, exponent):
    """Computes matrix powers and strips floating-point noise to keep columns orthonormal."""
    if isinstance(U, Operator):
        raw_power = U.power(exponent).data
    else:
        raw_power = np.linalg.matrix_power(np.array(U), exponent)

    dim = raw_power.shape[0]
    perfect_matrix = np.zeros((dim, dim), dtype=np.float64)

    for col in range(dim):
        max_row_idx = np.argmax(np.abs(raw_power[:, col]))
        perfect_matrix[max_row_idx, col] = 1.0

    return Operator(perfect_matrix)


In [ ]:
def recover_factors(a, N, r):
    """Given order r of a mod N, try to recover a non-trivial factor pair of N."""
    if r is None:
        return None

    if r % 2 != 0:
        return None

    x = pow(a, r // 2, N)

    if x == N - 1:
        return None

    factor_1 = math.gcd(x - 1, N)
    factor_2 = math.gcd(x + 1, N)

    if factor_1 == 1 or factor_1 == N:
        return None

    if factor_2 == 1 or factor_2 == N:
        return None

    # NOTE (fix): original swap here (`factor_1 = factor_2` instead of
    # `factor_1 = tempfactor`) discarded the larger value instead of keeping
    # it. Harmless downstream since factor_2 is always recomputed as
    # N // factor_1 below, but written correctly here for clarity.
    if factor_1 < factor_2:
        factor_1, factor_2 = factor_2, factor_1

    factor_2 = N // factor_1

    return factor_1, factor_2


In [ ]:
def build_iqpe_circuit(U, precision):
    """
    Build an Iterative Quantum Phase Estimation circuit for a unitary U.

    Parameters
    ----------
    U : Operator
        The unitary whose eigenphase is being estimated.
    precision : int
        Number of phase bits to estimate.

    Returns
    -------
    circuit : QuantumCircuit
    """
    n = U.num_qubits

    phase_qubit = QuantumRegister(1, "phase")
    system = QuantumRegister(n, "system")
    phase_bits = ClassicalRegister(precision, "phase_bits")

    circuit = QuantumCircuit(phase_qubit, system, phase_bits)

    # NOTE: system must be initialized to an eigenstate of U (or a state
    # with overlap on one). |1> is used here as a placeholder starting state
    # for Shor-style modular multiplication.
    circuit.x(system[0])

    for k in range(precision - 1, -1, -1):
        circuit.reset(phase_qubit[0])
        circuit.h(phase_qubit[0])

        # Controlled U^(2^k)
        exponent = 2 ** k
        U_power = power_of_unitary(U, exponent)
        matrix_data = U_power.data if hasattr(U_power, 'data') else U_power

        gate_instruction = UnitaryGate(matrix_data, check_input=False)
        controlled_U_power = gate_instruction.control(num_ctrl_qubits=1)

        circuit.append(controlled_U_power, [phase_qubit[0]] + list(system))

        # Phase feedback: classical conditional correction based on
        # previously measured bits. angle = -2*pi * 0.b_(k+1)...b_(m-1)
        if k < precision - 1:
            for j in range(k + 1, precision):
                angle = -math.pi / (2 ** (j - k))
                with circuit.if_test((phase_bits[j], 1)):
                    circuit.p(angle, phase_qubit[0])

        circuit.h(phase_qubit[0])
        circuit.measure(phase_qubit[0], phase_bits[k])

    return circuit


In [ ]:
def continued_fraction_order(phase, a, N):
    """
    Given an estimated phase ~= s / r, recover the order r using continued
    fractions. The candidate denominator is verified via a^r mod N == 1.
    """
    fraction = Fraction(phase).limit_denominator(N)
    candidate_r = fraction.denominator

    if pow(a, candidate_r, N) == 1:
        return candidate_r

    return None


In [ ]:
def bitstring_to_phase(bitstring):
    """Convert a binary phase estimate into a number in [0, 1).

    NOTE (fix): Qiskit prints the classical register with bit index 0 as the
    RIGHTMOST character of the string, but phase_bits[0] holds the
    MOST-significant fractional bit of the estimated phase (it's the last
    bit measured, using the smallest correction). Reading the raw string
    left-to-right as-is therefore inverts every phase estimate and made
    order recovery fail on every single run (verified: with the original
    code, Phase 1 on N=15 always returned r=None; reversing the string
    first correctly recovers r=2 for Gamma=4).
    """
    reversed_bits = bitstring[::-1]
    return int(reversed_bits, 2) / (2 ** len(reversed_bits))


def estimate_order_from_counts(counts, a, N):
    """Try each measured phase value, most frequent first, and recover an order."""
    for bitstring, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
        phase = bitstring_to_phase(bitstring)

        print(f"Measurement: {bitstring}")
        print(f"Estimated phase: {phase}")

        candidate_r = continued_fraction_order(phase, a, N)

        if candidate_r is not None:
            print(f"Candidate order r = {candidate_r}")
            return candidate_r

    return None


## 6. Phase 1 Driver

In [ ]:
def _get_backend_phase1(min_qubits):
    """Connect to IBM Quantum using the credentials from the Customization
    cell and pick the least busy backend that can fit this circuit."""
    kwargs = {"channel": "ibm_quantum_platform", "token": IBM_API_KEY}
    if IBM_INSTANCE_CRN:
        kwargs["instance"] = IBM_INSTANCE_CRN
    service = QiskitRuntimeService(**kwargs)
    return service.least_busy(simulator=False, operational=True, min_num_qubits=min_qubits)


def ApplyPhase1(N, precision=8, shots=1024):
    Gamma = ChooseGamma(N)
    U = BuildUnitary(Gamma, N)

    iqpe_circuit = build_iqpe_circuit(U=U, precision=precision)
    print(f"Logical circuit depth (fixed, uninformative): {iqpe_circuit.depth()}")

    # Hardware initialization with simulator fallback
    try:
        backend = _get_backend_phase1(precision)
        is_hardware = True
        print(f"Connected to IBM Quantum. Using backend: {backend.name}")
    except Exception as e:
        backend = AerSimulator()
        is_hardware = False
        print(f"Could not connect to hardware ({e}). Falling back to local AerSimulator.")

    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    transpiled_circuit = pm.run(iqpe_circuit)

    print(f"Transpiled depth (real cost): {transpiled_circuit.depth()}")
    print(f"Transpiled gate counts: {transpiled_circuit.count_ops()}")

    if is_hardware:
        print(f"Submitting to IBM Quantum ({backend.name}) via SamplerV2...")
        sys.stdout.flush()

        sampler = SamplerV2(backend)
        job = sampler.run([transpiled_circuit], shots=shots)
        result = job.result()

        pub_result = result[0]
        reg_name = list(pub_result.data.keys())[0]
        counts = pub_result.data[reg_name].get_counts()
    else:
        job = backend.run(transpiled_circuit, shots=shots)
        result = job.result()
        counts = result.get_counts()

    print("Counts:", counts)

    recovered_order = estimate_order_from_counts(counts, Gamma, N)
    factors = recover_factors(Gamma, N, recovered_order)

    if factors is not None:
        if factors[0] > factors[1]:
            factors = (factors[1], factors[0])
        print("Factors:", factors)
        return factors

    print(f"Factors: None (Shor's algorithm failed to find factors for order r = {recovered_order} on this run).")
    return None


## 7. Phase 2 Building Blocks

In [ ]:
def CreateEvenSuperposition(width, height):
    """Uniform superposition over all Height values, and over EVEN Width values
    (achieved by leaving the width LSB in |0>)."""
    QC, QRWidth, QRHeight = CreateSpaceDimensioned(width, height)

    # 1. Create superposition for ALL height values
    for i in range(height):
        QC.h(QRHeight[i])

    # 2. Create superposition for EVEN width values (skip the LSB)
    for i in range(1, width):
        QC.h(QRWidth[i])

    return QC


In [ ]:
"""
Hamming Weight resonance oracle for the fixed-point pi/3 Grover search.

Marks every (x, y) state in the search register satisfying:
    HW(x) XOR HW(y) == target_parity
where x is restricted to EVEN VALUES (LSB == 0), matching the constraint
imposed by CreateEvenSuperposition / GenerateKey.
"""

PI3_ANGLE = np.pi / 3  # Fixed pi/3 phase -- Grover (2005) fixed-point variant.


def ApplyHammingWeightOracle(QC, width, height, target_parity, angle=PI3_ANGLE):
    n = width + height
    x_qubits = list(range(width))
    y_qubits = list(range(width, n))

    for x in range(0, 2 ** width, 2):          # even x values only (LSB=0)
        hw_x = GiveHammingWeight(x)

        for y in range(2 ** height):
            hw_y = GiveHammingWeight(y)

            if (hw_x ^ hw_y) % 2 != target_parity:
                continue

            # Select |x, y> -- flip qubits that should be 0 for this state
            flipped = []
            for i, q in enumerate(x_qubits):
                if ((x >> i) & 1) == 0:
                    QC.x(q)
                    flipped.append(q)
            for i, q in enumerate(y_qubits):
                if ((y >> i) & 1) == 0:
                    QC.x(q)
                    flipped.append(q)

            # Apply pi/3 phase conditioned on all qubits being |1>
            controls = x_qubits + y_qubits
            if len(controls) == 1:
                QC.p(angle, controls[0])
            else:
                QC.append(MCPhaseGate(angle, num_ctrl_qubits=len(controls) - 1), controls)

            # Undo selection
            for q in flipped:
                QC.x(q)

    return QC


def BuildDiffuser(num_qubits, angle=PI3_ANGLE):
    """Fixed-point pi/3 Grover diffuser (inversion about the mean)."""
    qc = QuantumCircuit(num_qubits, name="Diffuser_pi3")
    qc.h(range(num_qubits))
    qc.x(range(num_qubits))
    qc.append(MCPhaseGate(angle, num_ctrl_qubits=num_qubits - 1), list(range(num_qubits)))
    qc.x(range(num_qubits))
    qc.h(range(num_qubits))
    return qc.to_instruction()


def BuildGroverCircuit(QC, width, height, target_parity, iterations=8, angle=PI3_ANGLE):
    """Applies `iterations` rounds of (oracle + diffuser) to QC."""
    n = width + height
    diffuser = BuildDiffuser(n, angle)

    for _ in range(iterations):
        QC = ApplyHammingWeightOracle(QC, width, height, target_parity, angle)
        QC.append(diffuser, list(range(n)))

    return QC


In [ ]:
"""
Builds a 2D probability heatmap over the (x, y) search-register lattice.
Assumes a NOISELESS classical simulator -- raw counts/shots are treated
directly as probabilities.
"""

def decode_bitstring(bitstring, width, height):
    """Decode a Qiskit measurement bitstring back into (x, y). Qubit i of the
    search register maps to character bitstring[n-1-i] (classical bit 0 is
    rightmost in Qiskit's returned string)."""
    rev = bitstring[::-1]  # rev[i] == bit measured on qubit i
    x_bits = rev[0:width]
    y_bits = rev[width:width + height]
    x = int(x_bits[::-1], 2) if width > 0 else 0
    y = int(y_bits[::-1], 2) if height > 0 else 0
    return x, y


def counts_to_probability_grid(counts, width_qubits, height_qubits, Width, Height):
    """Convert raw measurement counts into a 2D probability grid, clipped to
    the ACTUAL lattice size (Width x Height). Any measured (x, y) outside the
    real lattice is tallied as 'leaked' probability instead of being dropped."""
    total_shots = sum(counts.values())
    grid = np.zeros((Width, Height))
    leaked_p = 0.0

    for bitstring, count in counts.items():
        x, y = decode_bitstring(bitstring, width_qubits, height_qubits)
        p = count / total_shots
        if x < Width and y < Height:
            grid[x, y] += p
        else:
            leaked_p += p

    return grid, leaked_p


def PlotHeatmap(counts, width_qubits, height_qubits, Width, Height, key=None,
                 title="Key-State Preparation Probability", save_path=None):
    """Plot (and optionally save) a 2D heatmap of P(x, y) over the ACTUAL
    Width x Height lattice."""
    grid, leaked_p = counts_to_probability_grid(
        counts, width_qubits, height_qubits, Width, Height
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(grid.T, origin="lower", cmap="viridis", aspect="auto")

    ax.set_xlabel("x  (Width lattice index)")
    ax.set_ylabel("y  (Height lattice index)")
    ax.set_title(title)
    ax.set_xticks(range(Width))
    ax.set_yticks(range(Height))

    if key is not None:
        ax.scatter([key[0]], [key[1]], marker="*", s=280,
                   facecolors="none", edgecolors="red", linewidths=2,
                   label="Target key")
        ax.legend(loc="upper right")

    fig.colorbar(im, ax=ax, label="Probability")
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f"  Heatmap saved to {save_path}")

    print(f"  Probability outside the {Width}x{Height} lattice (leaked): {leaked_p:.4f}")

    plt.show()

    return fig, grid, leaked_p


In [ ]:
"""
Full error mitigation stack for the Phase 2 Grover search circuit, for use
on real IBM Quantum hardware. Not invoked by the default noiseless-simulator
Phase 2 flow below, but included/fixed here in case you want to run Phase 2
on real hardware.

Applies in order:
  1. Dynamical decoupling (XY4) + gate/measurement twirling -- on every job
  2. Zero-noise extrapolation (ZNE) via unitary folding      -- scale 1, 3, 5
  3. Matrix-free measurement error correction (M3)           -- per job
"""

def _key_to_bitstring(key, width, height):
    """Encode [key_x, key_y] as the bitstring the measurement register produces."""
    x_bits = format(key[0], f"0{width}b")[::-1]
    y_bits = format(key[1], f"0{height}b")[::-1]
    return (x_bits + y_bits)[::-1]


def _make_mitigated_sampler(backend):
    """SamplerV2 with DD (XY4) + gate/measurement twirling enabled."""
    sampler = SamplerV2(mode=backend)
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = "XY4"
    sampler.options.twirling.enable_gates = True
    sampler.options.twirling.enable_measure = True
    sampler.options.twirling.num_randomizations = "auto"
    return sampler


def _fold_circuit(circuit, scale_factor):
    """Unitary folding for ZNE: U -> U (U^-1 U)^k, scale_factor = 2k+1.
    Circuit must NOT contain measurements."""
    if scale_factor == 1:
        return circuit.copy()
    assert (scale_factor - 1) % 2 == 0, "scale_factor must be odd (1, 3, 5, ...)"
    k = (scale_factor - 1) // 2
    folded = circuit.copy()
    for _ in range(k):
        folded = folded.compose(circuit.inverse())
        folded = folded.compose(circuit)
    return folded


def _m3_correct(counts, backend, qubits):
    """Matrix-free M3 measurement error correction. Skipped for a local
    AerSimulator (nothing to correct, no noise model)."""
    if isinstance(backend, AerSimulator):
        total = sum(counts.values())
        return {bitstring: c / total for bitstring, c in counts.items()}

    import mthree
    mit = mthree.M3Mitigation(backend)
    mit.cals_from_system(qubits)
    quasi = mit.apply_correction(counts, qubits)
    return quasi.nearest_probability_distribution()


def _zne_extrapolate(scale_factors, probs, order=1):
    """Linear fit through (scale, P(target)) points, extrapolate to scale=0."""
    coeffs = np.polyfit(scale_factors, probs, order)
    return float(np.polyval(coeffs, 0)), coeffs


# NOTE (fix): default was `scale_factors=(1)`, a bare int (no trailing comma),
# not a tuple -- iterating over it in the `for sf in scale_factors:` loop
# below would raise TypeError. Fixed to `(1,)`.
def RunWithMitigation(GroverCircuit, backend, key, width, height,
                      scale_factors=(1,), shots=2048):
    """Run the Grover search circuit on real IBM hardware with the full
    DD + twirling + ZNE + M3 mitigation stack."""
    search_qubits = list(range(width + height))
    target_bitstring = _key_to_bitstring(key, width, height)
    print(f"  Target bitstring: {target_bitstring}")

    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    sampler = _make_mitigated_sampler(backend)

    target_probs        = []
    raw_counts_by_scale = []

    for sf in scale_factors:
        print(f"\n  [ZNE] Submitting scale_factor={sf}...")

        folded = _fold_circuit(GroverCircuit, sf)

        creg = ClassicalRegister(len(search_qubits), name="result")
        folded.add_register(creg)
        folded.measure(search_qubits, creg)

        isa = pm.run(folded)
        job = sampler.run([isa], shots=shots)
        print(f"  Job submitted. ID: {job.job_id()}")
        result = job.result()
        raw_counts = result[0].data.result.get_counts()

        mitigated = _m3_correct(raw_counts, backend, search_qubits)
        p_target  = mitigated.get(target_bitstring, 0.0)

        print(f"  scale={sf}: P(target)={p_target:.4f}  "
              f"raw shots at target={raw_counts.get(target_bitstring, 0)}")

        target_probs.append(p_target)
        raw_counts_by_scale.append(dict(raw_counts))

    zne_estimate, _ = _zne_extrapolate(list(scale_factors), target_probs)
    print(f"\n  ZNE extrapolated P(target key found) = {zne_estimate:.4f}")

    return zne_estimate, target_probs, raw_counts_by_scale


## 8. Phase 2 Driver

In [ ]:
GROVER_ITERATIONS = 8


# NOTE (fix): original had no try/except here, so it crashed outright unless
# real IBM Quantum credentials were supplied. Now falls back to AerSimulator,
# matching Phase 1's pattern.
def _get_backend_phase2(min_qubits=10):
    try:
        kwargs = {"channel": "ibm_quantum_platform", "token": IBM_API_KEY}
        if IBM_INSTANCE_CRN:
            kwargs["instance"] = IBM_INSTANCE_CRN
        service = QiskitRuntimeService(**kwargs)
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=min_qubits)
        print(f"  Connected to IBM Quantum. Using backend: {backend.name} ({backend.num_qubits} qubits)")
        is_hardware = True
    except Exception as e:
        backend = AerSimulator()
        print(f"  Could not connect to hardware ({e}). Falling back to local AerSimulator.")
        is_hardware = False
    sys.stdout.flush()
    return backend, is_hardware


def ApplyPhase2(Width, Height):
    print("\n--- [Phase 2] Parity-Restricted Grover Search ---")
    sys.stdout.flush()

    width_qubits = int(np.ceil(np.log2(Width + 1)))
    height_qubits = int(np.ceil(np.log2(Height + 1)))
    print(f"  Width={Width} ({width_qubits} qubits)  Height={Height} ({height_qubits} qubits)")
    sys.stdout.flush()

    Key = GenerateKey([Width, Height])
    if Key is None:
        print("  [WARNING] GenerateKey returned None -- no valid key for these dimensions.")
        sys.stdout.flush()
        return None

    print(f"  Key={Key}")
    sys.stdout.flush()

    target_parity = (GiveHammingWeight(Width) + GiveHammingWeight(Height)) % 2
    print(f"  Target parity: {target_parity}")
    sys.stdout.flush()

    Space = CreateEvenSuperposition(width_qubits, height_qubits)
    Space = BuildGroverCircuit(Space, width_qubits, height_qubits, target_parity,
                               iterations=GROVER_ITERATIONS)

    print(f"  Circuit: {Space.num_qubits} qubits, depth={Space.depth()}, "
          f"{GROVER_ITERATIONS} iterations")
    sys.stdout.flush()

    # Noiseless simulator (or hardware, if connected): measure directly;
    # the full mitigation stack (RunWithMitigation) is skipped here since
    # this default path targets AerSimulator. Swap in RunWithMitigation if
    # you're running on real hardware and want DD/ZNE/M3.
    search_qubits = list(range(width_qubits + height_qubits))
    creg = ClassicalRegister(len(search_qubits), name="result")
    Space.add_register(creg)
    Space.measure(search_qubits, creg)

    backend, is_hardware = _get_backend_phase2()
    transpiled_circuit = transpile(Space, backend=backend)

    job = backend.run(transpiled_circuit, shots=4096)
    counts = job.result().get_counts()

    heatmap_path = os.path.join(os.getcwd(), "phase2_heatmap.png")

    fig, probability_grid, leaked_p = PlotHeatmap(
        counts, width_qubits, height_qubits, Width, Height, key=Key,
        title=f"P(x, y) preparation -- Width={Width}, Height={Height}",
        save_path=heatmap_path,
    )

    p_target = probability_grid[Key[0], Key[1]]
    found = p_target >= 0.5

    print(f"\n  Target key   : {Key}")
    print(f"  P(target)    : {p_target:.4f}")
    print(f"  Result: {'KEY FOUND' if found else 'BEST CANDIDATE'}: {Key}")
    sys.stdout.flush()

    return {
        "Space": Space,
        "Key": Key,
        "Width": Width,
        "Height": Height,
        "width_qubits": width_qubits,
        "height_qubits": height_qubits,
        "target_parity": target_parity,
        "counts": counts,
        "probability_grid": probability_grid,
        "leaked_probability": leaked_p,
        "p_target": p_target,
        "found": found,
        "heatmap_fig": fig,
    }


## 9. Runner — Full Pipeline

Picks `N`, computes its dimensions and a demonstration key, then runs
Phase 1 (Shor's algorithm) up to `MAX_ATTEMPTS` times until it recovers
non-trivial factors, and finally runs Phase 2 (Grover search) over those
factors.


In [ ]:
N = GiveN()
print(f"Target Number N: {N}")
Dimensions = CreateDimension(N)
print(f"Target Dimension: {Dimensions}")
Key = GenerateKey(Dimensions)
print(f"Target Key: {Key}")

# Phase 1
MAX_ATTEMPTS = 20
factors_discovered = None

for attempt in range(MAX_ATTEMPTS):
    print(f"\n--- [Phase 1] Shor's Execution Attempt {attempt + 1}/{MAX_ATTEMPTS} ---")

    try:
        factors_discovered = ApplyPhase1(N)
        if factors_discovered is not None:
            print(f"\n🎉 Success! Found non-trivial factors for {N}: {factors_discovered}")
            break
    except ValueError as e:
        print(f" Skipping this attempt: {e}")
        continue
else:
    print(f"\n Failed to isolate factors across {MAX_ATTEMPTS} attempts.")

Width, Height = factors_discovered

# Phase 2
Space = ApplyPhase2(Width, Height)
